In [1]:
import sys
sys.path.append("../")

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from pathlib import Path
from tqdm.auto import tqdm

In [3]:
from utils.imagenet_txt_dataset import ImageNetJsonDataset

In [4]:
IMG_SIZE_PRE = 1022 # Must be divisible by 14.
IMG_SIZE = 1024 # Must be divisible by 16
DEVICE = "cuda:2"

OUTPUT_DIR = Path("../data/analysis")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

In [5]:
dinov2_transform_pre = transforms.Compose([
    transforms.CenterCrop((IMG_SIZE_PRE, IMG_SIZE_PRE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dinov2_transform = transforms.Compose([
    transforms.CenterCrop((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [6]:
us_dataset_old_scheme_pre = ImageNetJsonDataset(
    json_file="../data/soilsense/september/anns/unevenSurface.json",
    root="../data/soilsense/september/",
    image_key="img_path_5",
    transform=dinov2_transform_pre
)
us_dataset_new_scheme_pre = ImageNetJsonDataset(
    json_file="../data/soilsense/september/anns/unevenSurfaceNewSchemeBinarized.json",
    root="../data/soilsense/september/",
    image_key="img_path_5",
    transform=dinov2_transform_pre
)


um_dataset_old_scheme_pre = ImageNetJsonDataset(
    json_file="../data/soilsense/october/anns/unmergedSurface.json",
    root="../data/soilsense/october/",
    image_key="img_path_4",
    transform=dinov2_transform_pre
)
um_dataset_new_scheme_pre = ImageNetJsonDataset(
    json_file="../data/soilsense/october/anns/unmergedSurfaceNewSchemeBinary.json",
    root="../data/soilsense/october/",
    image_key="img_path_4",
    transform=dinov2_transform_pre
)

In [7]:
us_dataset_old_scheme = ImageNetJsonDataset(
    json_file="../data/soilsense/september/anns/unevenSurface.json",
    root="../data/soilsense/september/",
    image_key="img_path_5",
    transform=dinov2_transform
)
us_dataset_new_scheme = ImageNetJsonDataset(
    json_file="../data/soilsense/september/anns/unevenSurfaceNewSchemeBinarized.json",
    root="../data/soilsense/september/",
    image_key="img_path_5",
    transform=dinov2_transform
)


um_dataset_old_scheme = ImageNetJsonDataset(
    json_file="../data/soilsense/october/anns/unmergedSurface.json",
    root="../data/soilsense/october/",
    image_key="img_path_4",
    transform=dinov2_transform
)
um_dataset_new_scheme = ImageNetJsonDataset(
    json_file="../data/soilsense/october/anns/unmergedSurfaceNewSchemeBinary.json",
    root="../data/soilsense/october/",
    image_key="img_path_4",
    transform=dinov2_transform
)

In [8]:
us_dataset_old_scheme_pre_loader = DataLoader(us_dataset_old_scheme_pre, batch_size=4, shuffle=False, num_workers=0)
us_dataset_new_scheme_pre_loader = DataLoader(us_dataset_new_scheme_pre, batch_size=4, shuffle=False, num_workers=0)
um_dataset_old_scheme_pre_loader = DataLoader(um_dataset_old_scheme_pre, batch_size=4, shuffle=False, num_workers=0)
um_dataset_new_scheme_pre_loader = DataLoader(um_dataset_new_scheme_pre, batch_size=4, shuffle=False, num_workers=0)

us_dataset_old_scheme_loader = DataLoader(us_dataset_old_scheme, batch_size=4, shuffle=False, num_workers=0)
us_dataset_new_scheme_loader = DataLoader(us_dataset_new_scheme, batch_size=4, shuffle=False, num_workers=0)
um_dataset_old_scheme_loader = DataLoader(um_dataset_old_scheme, batch_size=4, shuffle=False, num_workers=0)
um_dataset_new_scheme_loader = DataLoader(um_dataset_new_scheme, batch_size=4, shuffle=False, num_workers=0)

In [9]:
def load_dino_pretrained(model_name):
    model = torch.hub.load('facebookresearch/dinov2', model_name)
    model = model.to(DEVICE)
    model.eval()
    return model

def load_dino_adapted(model_name):
    pass


def visualize_features(features, labels, class_names, method='tsne', title='DINOv2 Features', save: bool = False):
    """
    Visualize features in 2D.
    """
    print(f"\n🎨 Computing {method.upper()} embedding...")
    
    if method == 'tsne':
        perplexity = min(30, len(features) - 1)
        reducer = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
    else:  # PCA
        reducer = PCA(n_components=2, random_state=42)
    
    embeddings = reducer.fit_transform(features)
    
    # Plot
    plt.figure(figsize=(10, 8))
    colors = plt.cm.Set1(np.linspace(0, 1, len(class_names)))
    
    for cls_id in range(len(class_names)):
        mask = labels == cls_id
        plt.scatter(embeddings[mask, 0], embeddings[mask, 1],
                   c=[colors[cls_id]], label=class_names[cls_id],
                   alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
    
    plt.xlabel(f'{method.upper()} Dimension 1', fontsize=12)
    plt.ylabel(f'{method.upper()} Dimension 2', fontsize=12)
    plt.title(f'{title} ({method.upper()})', fontsize=14, fontweight='bold')
    plt.legend(title='Classes', loc='best', fontsize=11)
    plt.grid(True, alpha=0.3)

    if save:
        filename = f"dinov2_{method}_visualization.png"
        plt.savefig(OUTPUT_DIR / filename, dpi=150, bbox_inches='tight')
        print(f"💾 Saved: {config.OUTPUT_DIR / filename}")
    
    plt.show()
    
    
    return embeddings


def compare_feature_space(train_feat, train_lbl, test_feat, test_lbl, class_names, save: bool = False):
    """
    Compare train and test feature distributions using PCA.
    """
    # Combine for joint PCA
    combined = np.vstack([train_feat, test_feat])
    
    pca = PCA(n_components=2, random_state=42)
    combined_2d = pca.fit_transform(combined)
    
    train_2d = combined_2d[:len(train_feat)]
    test_2d = combined_2d[len(train_feat):]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    colors = plt.cm.Set1(np.linspace(0, 1, len(class_names)))
    
    # Training
    for cls_id in range(len(class_names)):
        mask = train_lbl == cls_id
        axes[0].scatter(train_2d[mask, 0], train_2d[mask, 1],
                       c=[colors[cls_id]], label=class_names[cls_id],
                       alpha=0.7, s=50)
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].set_title('Training Set Features', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Test
    for cls_id in range(len(class_names)):
        mask = test_lbl == cls_id
        axes[1].scatter(test_2d[mask, 0], test_2d[mask, 1],
                       c=[colors[cls_id]], label=class_names[cls_id],
                       alpha=0.7, s=50)
    axes[1].set_xlabel('PC1')
    axes[1].set_ylabel('PC2')
    axes[1].set_title('Test Set Features', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(f'DINOv2 Feature Space Comparison (PCA, Var: {sum(pca.explained_variance_ratio_)*100:.1f}%)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save:
        plt.savefig(OUTPUT_DIR / 'dinov2_train_test_comparison.png', dpi=150, bbox_inches='tight')
        print(f"💾 Saved: {OUTPUT_DIR / 'dinov2_train_test_comparison.png'}")
    
    plt.show()
    


def analyze_features(features, labels, class_names, split_name):
    """
    Analyze feature statistics.
    """
    print(f"\n{'='*60}")
    print(f"📊 FEATURE ANALYSIS: {split_name}")
    print(f"{'='*60}")
    
    print(f"\n📐 Overall Statistics:")
    print(f"   Shape: {features.shape}")
    print(f"   Mean: {features.mean():.4f}")
    print(f"   Std: {features.std():.4f}")
    print(f"   Min: {features.min():.4f}")
    print(f"   Max: {features.max():.4f}")
    
    # L2 norms
    norms = np.linalg.norm(features, axis=1)
    print(f"\n📏 L2 Norm Statistics:")
    print(f"   Mean: {norms.mean():.4f}")
    print(f"   Std: {norms.std():.4f}")
    
    # Per-class analysis
    print(f"\n🏷️ Per-Class Statistics:")
    for cls_id in range(len(class_names)):
        mask = labels == cls_id
        if mask.sum() > 0:
            cls_features = features[mask]
            cls_norms = norms[mask]
            print(f"   {class_names[cls_id]}:")
            print(f"      Count: {mask.sum()}")
            print(f"      Mean L2 Norm: {cls_norms.mean():.4f}")
            print(f"      Feature Mean: {cls_features.mean():.4f}")

# Feature distribution plot
def plot_feature_distributions(features, labels, class_names, save: bool = False):
    """
    Plot feature value distributions.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Overall feature distribution
    axes[0].hist(features.flatten(), bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Feature Value')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Overall Feature Distribution', fontweight='bold')
    axes[0].axvline(features.mean(), color='red', linestyle='--', label=f'Mean: {features.mean():.2f}')
    axes[0].legend()
    
    # L2 norm distribution
    norms = np.linalg.norm(features, axis=1)
    axes[1].hist(norms, bins=30, alpha=0.7, color='coral', edgecolor='black')
    axes[1].set_xlabel('L2 Norm')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Feature L2 Norm Distribution', fontweight='bold')
    axes[1].axvline(norms.mean(), color='red', linestyle='--', label=f'Mean: {norms.mean():.2f}')
    axes[1].legend()
    
    # Per-class norm distribution
    colors = plt.cm.Set1(np.linspace(0, 1, len(class_names)))
    for cls_id in range(len(class_names)):
        mask = labels == cls_id
        if mask.sum() > 0:
            cls_norms = norms[mask]
            axes[2].hist(cls_norms, bins=20, alpha=0.5, label=class_names[cls_id], color=colors[cls_id])
    axes[2].set_xlabel('L2 Norm')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('L2 Norm by Class', fontweight='bold')
    axes[2].legend()
    
    plt.tight_layout()
    if save:
        plt.savefig(OUTPUT_DIR / 'dinov2_feature_distributions.png', dpi=150, bbox_inches='tight')
        print(f"💾 Saved: {OUTPUT_DIR / 'dinov2_feature_distributions.png'}")

    plt.show()
    

@torch.no_grad()
def extract_dinov2_features(model, dataloader, desc="Extracting features"):
    """
    Extract DINOv2 CLS token features from all images.
    
    Returns:
        features: numpy array of shape (N, feature_dim)
        labels: numpy array of labels
        paths: list of image paths
    """
    model.eval()
    
    all_features = []
    all_labels = []
    
    for images, labels in tqdm(dataloader, desc=desc):
        images = images.to(DEVICE)
        
        # Extract CLS token features
        features = model(images)
        
        all_features.append(features.cpu().numpy())
        all_labels.extend(labels.numpy())
    
    features = np.concatenate(all_features, axis=0)
    labels = np.array(all_labels)
    
    return features, labels

# Class distribution in features
def plot_class_distribution(labels, class_names, title, save: bool = False):
    counts = Counter(labels)
    
    plt.figure(figsize=(8, 5))
    bars = plt.bar(class_names, [counts[i] for i in range(len(class_names))],
                   color=plt.cm.Set2(np.linspace(0, 1, len(class_names))),
                   edgecolor='black')
    
    for bar, count in zip(bars, [counts[i] for i in range(len(class_names))]):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(count), ha='center', va='bottom', fontweight='bold')
    
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.title(title, fontweight='bold')
    plt.tight_layout()
    if save:
        plt.savefig(OUTPUT_DIR / 'dinov2_class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [10]:
model_pre = load_dino_pretrained("dinov2_vitl14")

In [ ]:
# Extract from all splits
us_dataset_old_scheme_pre_features, us_dataset_old_scheme_pre_labels = extract_dinov2_features(
    model_pre, us_dataset_old_scheme_pre_loader, desc="us_dataset_old_scheme"
)

us_dataset_old_scheme:   3%|██▎                                                                    | 50/1563 [02:53<1:25:20,  3.38s/it]

In [ ]:






val_features, val_labels, val_paths = extract_dinov2_features(
    dinov2_model, val_loader, desc="Validation set"
)

test_features, test_labels, test_paths = extract_dinov2_features(
    dinov2_model, test_loader, desc="Test set"
)

plot_feature_distributions(train_features, train_labels, config.CLASS_NAMES)


# t-SNE visualization
tsne_embeddings = visualize_features(
    train_features, train_labels, config.CLASS_NAMES,
    method='tsne', title='DINOv2 Training Features'
)

# PCA visualization
pca_embeddings = visualize_features(
    train_features, train_labels, config.CLASS_NAMES,
    method='pca', title='DINOv2 Training Features'
)
compare_feature_space(train_features, train_labels, test_features, test_labels, config.CLASS_NAMES)










plot_class_distribution(train_labels, config.CLASS_NAMES, 'Training Set Class Distribution')

# ==================================
# FINAL SUMMARY
# ==================================
print("\n" + "=" * 70)
print("📋 DINOv2 FEATURE EXTRACTION - COMPLETE SUMMARY")
print("=" * 70)
print(f"""
🤖 MODEL INFORMATION:
   ├── Model: {config.MODEL_NAME}
   ├── Parameters: {sum(p.numel() for p in dinov2_model.parameters()):,}
   └── Feature Dimension: {config.FEATURE_DIM}

📊 EXTRACTED FEATURES:
   ┌─────────────┬────────────────────┬──────────────┐
   │   Split     │      Shape         │   Samples    │
   ├─────────────┼────────────────────┼──────────────┤
   │   Train     │ {str(train_features.shape):18s} │ {len(train_features):>10,d}   │
   │   Val       │ {str(val_features.shape):18s} │ {len(val_features):>10,d}   │
   │   Test      │ {str(test_features.shape):18s} │ {len(test_features):>10,d}   │
   └─────────────┴────────────────────┴──────────────┘

📈 FEATURE STATISTICS:
   ├── Mean: {train_features.mean():.4f}
   ├── Std:  {train_features.std():.4f}
   └── Mean L2 Norm: {np.linalg.norm(train_features, axis=1).mean():.4f}
""")



# Pretrained DinoV2